In [ ]:
from src.methods import run_awb
from cfgs import ALL_CONFIGS
import src.test as test
from src.utils import config, isp
import matplotlib.pyplot as plt

cfg_3_t = config.AWBConfig(
    debug=False,
    fast=False,
    awb_method=3,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.97,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.2,
    downsample=True,
    downsample_scale=0.3,
    region_method='superpixels',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.1,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-3},
    smooth2_method='none',
    smooth2_kwargs={},
)

img = test.get_image(ill_idx=0, mode='multi')
cfg = cfg_3_t
# running once to warm up
print("Warm-up run...")
result = run_awb(img, cfg, cfg.awb_method)

# Now measuring time
print("Measuring run...")
result = run_awb(img, cfg, cfg.awb_method)

plt.figure(figsize=(10,10))
plt.imshow(isp.gamma_correction(result['image']))
plt.show()

# saving all results individually

## using the base cfg

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from results.save_results import run_awb_and_save
from src import methods, config_io, test
from src.utils import isp, config
import cfgs

all_cfgs = cfgs.get_base_cfgs()

parent_dir = 'results/individual_images_base_cfgs'
mode = 'multi'

for cfg in all_cfgs:
    if cfg.region_method == 'tiles':
        reg = 't'
    else:
        reg = 's'

    for ill_idx, ill in enumerate(test.ILLUMINANTS):
        img = test.get_image(ill_idx, mode=mode)

        run_awb_and_save(ill_idx, mode, img, cfg, f'{parent_dir}/results_{cfg.awb_method}_{reg}', f'results_{cfg.awb_method}_{reg}_{ill}')

## using special cfgs

In [ ]:
from results.save_results import run_awb_and_save
import numpy as np
import matplotlib.pyplot as plt

from src import methods, config_io, test
from src.utils import isp, config
import cfgs

parent_dir = 'results/individual_images_new'
mode = 'multi'

for cfg in cfgs.ALL_CONFIGS:
    if cfg.region_method == 'tiles':
        reg = 't'
    else:
        reg = 's'

    for ill_idx, ill in enumerate(test.ILLUMINANTS):
        img = test.get_image(ill_idx, mode=mode)

        run_awb_and_save(ill_idx, mode, img, cfg, f'{parent_dir}/results_{cfg.awb_method}_{reg}', f'results_{cfg.awb_method}_{reg}_{ill}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src import methods, config_io, test
from src.utils import isp, config

NUM_METHODS = 4
NUM_ILLUMINANTS = 5

ILLUMINANTS = ['A', 'CWF', 'D65', 'HZ', 'TL84']
SINGLE_FILES = [f'capture/lightbox/img_{ill}.npy' for ill in ILLUMINANTS]
MULTI_FILES = [f'capture/lightbox/img_multi_{ill}.npy' for ill in ILLUMINANTS]

MODES = ['single', 'multi_big', 'multi_small']
REGIONS = ['sp', 't']


def get_config(method: int, region: str):
    filename = f'config_method{method}_{region}_lab-multi.json'
    return config_io.load_config_json(filename)

# ------------------------------------------------------------------

def get_image(ill_idx: int, mode: str):
    if mode == 'single':
        img_file = SINGLE_FILES[ill_idx]
    elif mode == 'multi':
        img_file = MULTI_FILES[ill_idx]
    else:
        raise ValueError(f'Unknown mode: {mode}')
    img = np.load(img_file)
    img = isp.black_level_correction(img)
    img = isp.demosaic(img)
    return img

# ------------------------------------------------------------------    
cfg = config.AWBConfig(
    debug=True,
    awb_method=3,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.1,
    downsample=True,
    downsample_scale=0.2,
    region_method='tiles',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.05,
    smooth1_method='none',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-4},
    smooth2_method='none',
    smooth2_kwargs={},
)
# ------------------------------------------------------------------
img = get_image(ill_idx=0, mode='multi')
result = methods.run_awb(img, cfg, cfg.awb_method)
# plt.imshow(isp.gamma_correction(result['image']))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src import methods, config_io, test
from src.utils import isp, config

# ------------------------------------------------------------------
# Configuration helpers
# ------------------------------------------------------------------

NUM_METHODS = 4
NUM_ILLUMINANTS = 5

ILLUMINANTS = ['A', 'CWF', 'D65', 'HZ', 'TL84']
SINGLE_FILES = [f'capture/lightbox/img_{ill}.npy' for ill in ILLUMINANTS]
MULTI_FILES = [f'capture/lightbox/img_multi_{ill}.npy' for ill in ILLUMINANTS]

MODES = ['single', 'multi_big', 'multi_small']
REGIONS = ['sp', 't']


def get_config(method: int, region: str):
    filename = f'config_method{method}_{region}_lab-multi.json'
    return config_io.load_config_json(filename)

# ------------------------------------------------------------------

def get_image(ill_idx: int, mode: str):
    if mode == 'single':
        img_file = SINGLE_FILES[ill_idx]
    elif mode == 'multi':
        img_file = MULTI_FILES[ill_idx]
    else:
        raise ValueError(f'Unknown mode: {mode}')
    img = np.load(img_file)
    img = isp.black_level_correction(img)
    img = isp.demosaic(img)
    return img



# all white points all illuminants all algorithms

In [ ]:
from src import methods, test
from src import test

def euclidean_distance(wp):
    return np.linalg.norm(wp - np.array([1.0, 1.0]))

def compute_average_distance(wps_big, wps_small):
    wps_big_rb = np.zeros((test.NUM_ILLUMINANTS, 2)) # White points in (R/G, B/G) space
    wps_small_rb = np.zeros((test.NUM_ILLUMINANTS, 2))
    wps_big_rb[:,0] = wps_big[:,0] / wps_big[:,1]
    wps_big_rb[:,1] = wps_big[:,2] / wps_big[:,1]
    wps_small_rb[:,0] = wps_small[:,0] / wps_small[:,1]
    wps_small_rb[:,1] = wps_small[:,2] / wps_small[:,1]

    distances = np.zeros((test.NUM_ILLUMINANTS, 2)) # Distances to pure white point, columns: big, small, row: illuminants
    for i in range(test.NUM_ILLUMINANTS):
        distances[i,0] = euclidean_distance(wps_big_rb[i])
        distances[i,1] = euclidean_distance(wps_small_rb[i])

    avg_distance_big = np.mean(distances[:,0])
    avg_distance_small = np.mean(distances[:,1])
    # print(f'Average Euclidean Distance to Pure White Point - Big: {avg_distance_big:.4f}, Small: {avg_distance_small:.4f}')

    avg_distance = (avg_distance_big + avg_distance_small) / 2
    # print(f'Overall Average Euclidean Distance to Pure White Point: {avg_distance:.4f}')
    return distances, avg_distance_big, avg_distance_small, avg_distance

def get_wps(mode, cfg):
    wps_big = np.zeros((test.NUM_ILLUMINANTS, 3))
    wps_small = np.zeros((test.NUM_ILLUMINANTS, 3))

    for ill_idx, ill in enumerate(test.ILLUMINANTS):
        img = test.get_image(ill_idx, mode)

        result = methods.run_awb(img, cfg, cfg.awb_method)
        img_awb = result['image']

        if mode == 'multi':
            wps_small[ill_idx] = test.get_wp_multi_small(img_awb, ill_idx)
            wps_big[ill_idx] = test.get_wp_multi_big(img_awb, ill_idx)
        elif mode == 'single':
            wps_big[ill_idx] = test.get_wp_single(img_awb, ill_idx)

    return wps_big, wps_small

# def get_wps_rb(mode, cfg):
#     wps_big = np.zeros((test.NUM_ILLUMINANTS, 3))
#     wps_small = np.zeros((test.NUM_ILLUMINANTS, 3))

#     for ill_idx, ill in enumerate(test.ILLUMINANTS):
#         img = test.get_image(ill_idx, mode)

#         result = methods.run_awb(img, cfg, cfg.awb_method)
#         img_awb = result['image']

#         if mode == 'multi':
#             wps_small[ill_idx] = test.get_wp_multi_small(img_awb, ill_idx)
#             wps_big[ill_idx] = test.get_wp_multi_big(img_awb, ill_idx)
#         elif mode == 'single':
#             wps_big[ill_idx] = test.get_wp_single(img_awb, ill_idx)

#     wps_big_rb = np.zeros((test.NUM_ILLUMINANTS, 2)) # White points in (R/G, B/G) space
#     wps_small_rb = np.zeros((test.NUM_ILLUMINANTS, 2))
#     wps_big_rb[:,0] = wps_big[:,0] / wps_big[:,1]
#     wps_big_rb[:,1] = wps_big[:,2] / wps_big[:,1]
#     if mode == 'multi':
#         wps_small_rb[:,0] = wps_small[:,0] / wps_small[:,1]
#         wps_small_rb[:,1] = wps_small[:,2] / wps_small[:,1]
#     return wps_big_rb,  wps_small_rb



# per-illuminant distance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src import test, methods
import cfgs

def euclidean_distance(wp):
    return np.linalg.norm(wp - np.array([1.0, 1.0]))

def compute_average_distance(wps_big, wps_small):
    wps_big_rb = np.zeros((test.NUM_ILLUMINANTS, 2)) # White points in (R/G, B/G) space
    wps_small_rb = np.zeros((test.NUM_ILLUMINANTS, 2))
    wps_big_rb[:,0] = wps_big[:,0] / wps_big[:,1]
    wps_big_rb[:,1] = wps_big[:,2] / wps_big[:,1]
    wps_small_rb[:,0] = wps_small[:,0] / wps_small[:,1]
    wps_small_rb[:,1] = wps_small[:,2] / wps_small[:,1]

    distances = np.zeros((test.NUM_ILLUMINANTS, 2)) # Distances to pure white point, columns: big, small, row: illuminants
    for i in range(test.NUM_ILLUMINANTS):
        distances[i,0] = euclidean_distance(wps_big_rb[i])
        distances[i,1] = euclidean_distance(wps_small_rb[i])

    avg_distance_big = np.mean(distances[:,0])
    avg_distance_small = np.mean(distances[:,1])
    # print(f'Average Euclidean Distance to Pure White Point - Big: {avg_distance_big:.4f}, Small: {avg_distance_small:.4f}')

    avg_distance = (avg_distance_big + avg_distance_small) / 2
    # print(f'Overall Average Euclidean Distance to Pure White Point: {avg_distance:.4f}')
    return distances, avg_distance_big, avg_distance_small, avg_distance

def get_wps(mode, cfg):
    wps_big = np.zeros((test.NUM_ILLUMINANTS, 3))
    wps_small = np.zeros((test.NUM_ILLUMINANTS, 3))

    for ill_idx, ill in enumerate(test.ILLUMINANTS):
        img = test.get_image(ill_idx, mode)

        result = methods.run_awb(img, cfg, cfg.awb_method)
        img_awb = result['image']

        if mode == 'multi':
            wps_small[ill_idx] = test.get_wp_multi_small(img_awb, ill_idx)
            wps_big[ill_idx] = test.get_wp_multi_big(img_awb, ill_idx)
        elif mode == 'single':
            wps_big[ill_idx] = test.get_wp_single(img_awb, ill_idx)

    return wps_big, wps_small
# ------------------------------------------------------------------

USE_BASE_CFGS = False

if USE_BASE_CFGS:
    all_cfgs = cfgs.get_base_cfgs()
    parent_dir = 'results/wp/base'
else:
    all_cfgs = cfgs.ALL_CONFIGS
    parent_dir = 'results/wp/custom'

# ------------------------------------------------------------------
mode = 'multi'  # 'single' or 'multi'
num_methods = len(all_cfgs)

all_distances = np.zeros((len(all_cfgs), test.NUM_ILLUMINANTS, 2)) # methods, illuminants, big/small

for idx, cfg in enumerate(all_cfgs):
    wps_big, wps_small = get_wps(mode, cfg)
    distances , avg_distance_big, avg_distance_small, avg_distance = compute_average_distance(wps_big, wps_small)
    all_distances[idx] = distances

mean_big = np.zeros(num_methods)
median_big = np.zeros(num_methods)
mean_small = np.zeros(num_methods)
median_small = np.zeros(num_methods)
for m in range(num_methods):
    d = all_distances[m]  # shape: (illuminants, 2)
    mean_big[m] = np.mean(d[:, 0])
    median_big[m] = np.median(d[:, 0])
    mean_small[m] = np.mean(d[:, 1])
    median_small[m] = np.median(d[:, 1])
# ------------------------------------------------------------------
# Create per-illuminant distance tables
# Assuming ILLUMINANTS is defined, otherwise create generic column names
try:
    illuminant_names = [f'{ill}' for ill in test.ILLUMINANTS]
except:
    illuminant_names = [f'Ill_{i}' for i in range(test.NUM_ILLUMINANTS)]

method_names = [f'{test.alg_names[cfg.awb_method]} - {test.reg_names[cfg.region_method]}' for cfg in all_cfgs]

# Table for BIG illuminant
print("\n" + "="*80)
print("WHITE POINT DISTANCES - PRIMARY ILLUMINANT")
print("="*80)

data_big = {}
data_big['Algorithm'] = method_names
for ill_idx in range(test.NUM_ILLUMINANTS):
    data_big[illuminant_names[ill_idx]] = all_distances[:, ill_idx, 0]

# Add mean, median, std columns
data_big['Mean'] = mean_big
data_big['Median'] = median_big
data_big['Std'] = [np.std(all_distances[m, :, 0]) for m in range(len(all_cfgs))]


df_big = pd.DataFrame(data_big)
print(df_big.round(4).to_string(index=False))

# Table for SMALL illuminant
print("\n" + "="*80)
print("WHITE POINT DISTANCES - SECONDARY ILLUMINANT")
print("="*80)

data_small = {}
data_small['Algorithm'] = method_names
for ill_idx in range(test.NUM_ILLUMINANTS):
    data_small[illuminant_names[ill_idx]] = all_distances[:, ill_idx, 1]

# Add mean, median, and std columns
data_small['Mean'] = mean_small
data_small['Median'] = median_small
data_small['Std'] = [np.std(all_distances[m, :, 1]) for m in range(len(all_cfgs))]

df_small = pd.DataFrame(data_small)
print(df_small.round(4).to_string(index=False))

# Degradation ratio table (Small/Big per illuminant)
# print("\n" + "="*80)
# print("DEGRADATION RATIO (SECONDARY/PRIMARY) PER ILLUMINANT")
# print("="*80)

data_ratio = {}
data_ratio['Algorithm'] = method_names
for ill_idx in range(test.NUM_ILLUMINANTS):
    ratios = all_distances[:, ill_idx, 1] / (all_distances[:, ill_idx, 0] + 1e-6)
    data_ratio[illuminant_names[ill_idx]] = ratios

# Add mean ratio
data_ratio['Mean Ratio'] = [mean_small[m] / (mean_big[m] + 1e-6) for m in range(len(all_cfgs))]

df_ratio = pd.DataFrame(data_ratio)
# print(df_ratio.round(2).to_string(index=False))

# # Identify worst cases
# print("\n" + "="*80)
# print("WORST CASES (Error > 0.30)")
# print("="*80)

# worst_cases = []
# for m_idx in range(len(all_cfgs)):
#     for ill_idx in range(test.NUM_ILLUMINANTS):
#         err_big = all_distances[m_idx, ill_idx, 0]
#         err_small = all_distances[m_idx, ill_idx, 1]
        
#         if err_big > 0.30:
#             worst_cases.append({
#                 'Algorithm': method_names[m_idx],
#                 'Illuminant': illuminant_names[ill_idx],
#                 'Type': 'Primary',
#                 'Error': err_big
#             })
#         if err_small > 0.30:
#             worst_cases.append({
#                 'Algorithm': method_names[m_idx],
#                 'Illuminant': illuminant_names[ill_idx],
#                 'Type': 'Secondary',
#                 'Error': err_small
#             })

# if worst_cases:
#     df_worst = pd.DataFrame(worst_cases).sort_values('Error', ascending=False)
#     print(df_worst.to_string(index=False))
# else:
#     print("No errors exceeding 0.30 found!")

# ------------------------------------------------------------------
# PLOTTING
# ------------------------------------------------------------------
# Heatmap visualization

# Heatmap 1: Big illuminant
plt.figure(figsize=(8,5))
im1 = plt.imshow(all_distances[:, :, 0], cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.3)
plt.xlabel('Primary Illuminant', fontsize=12)
plt.ylabel('Algorithm', fontsize=12)
plt.title('White Point Errors - Primary Illuminant', fontsize=14)
plt.xticks(range(test.NUM_ILLUMINANTS), illuminant_names)
plt.yticks(range(num_methods), method_names)

# Add text annotations
for m in range(num_methods):
    for ill in range(test.NUM_ILLUMINANTS):
        val = all_distances[m, ill, 0]
        color = 'black' # 'white' if val > 0.15 else 'black'
        plt.text(ill, m, f'{val:.3f}', ha='center', va='center', 
                    color=color, fontsize=9)

plt.colorbar(im1, label='Euclidean Distance')

plt.tight_layout()
plt.savefig(f'{parent_dir}/wp_error_heatmap_primary_illuminant.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Heatmap 2: Small illuminant
plt.figure(figsize=(8,5))
im2 = plt.imshow(all_distances[:, :, 1], cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.5)
plt.xlabel('Primary Illuminant', fontsize=12)
plt.ylabel('Algorithm', fontsize=12)
plt.title('White Point Errors - Secondary Illuminant', fontsize=14)
plt.xticks(range(test.NUM_ILLUMINANTS), illuminant_names)
plt.yticks(range(num_methods), method_names)

# Add text annotations
for m in range(num_methods):
    for ill in range(test.NUM_ILLUMINANTS):
        val = all_distances[m, ill, 1]
        color = 'black' # 'white' if val > 0.25 else 'black'
        plt.text(ill, m, f'{val:.3f}', ha='center', va='center', 
                    color=color, fontsize=9)

plt.colorbar(im2, label='Euclidean Distance')

plt.tight_layout()
plt.savefig(f'{parent_dir}/wp_error_heatmap_secondary_illuminant.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Export tables to CSV for thesis
df_big.round(3).to_csv(f'{parent_dir}/wp_primary_illuminant.csv', index=False)
df_small.round(3).to_csv(f'{parent_dir}/wp_secondary_illuminant.csv', index=False)
df_ratio.round(2).to_csv(f'{parent_dir}/wp_degradation_ratio.csv', index=False)
print("\nTables exported to CSV files for thesis")

## convert to latex tables

In [ ]:
USE_BASE_CFGS = False

if USE_BASE_CFGS:
    parent_dir = 'results/wp/base'
else:
    parent_dir = 'results/wp/custom'

from results.csv_to_latex import csv_to_latex_table

csv_to_latex_table(f'{parent_dir}/wp_primary_illuminant.csv', 
    caption='White Point Distances - Primary Illuminant', 
    label='tab:wp_primary_illuminant', 
    float_format='.3f')
print()
csv_to_latex_table(f'{parent_dir}/wp_secondary_illuminant.csv', 
    caption='White Point Distances - Secondary Illuminant', 
    label='tab:wp_secondary_illuminant', 
    float_format='.3f')

# per illuminant

In [ ]:
# Per-illuminant analysis
import matplotlib.pyplot as plt
import numpy as np

# Compute per-illuminant statistics
illuminant_stats = np.zeros((NUM_ILLUMINANTS, num_methods, 2))  # illuminants x methods x (big/small)

for ill_idx in range(NUM_ILLUMINANTS):
    for m_idx in range(num_methods):
        illuminant_stats[ill_idx, m_idx, 0] = all_distances[m_idx, ill_idx, 0]  # big
        illuminant_stats[ill_idx, m_idx, 1] = all_distances[m_idx, ill_idx, 1]  # small

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('White Point Error Analysis by Illuminant', fontsize=16)

# 1. Heatmap for big patches
ax1 = axes[0, 0]
im1 = ax1.imshow(illuminant_stats[:, :, 0].T, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.3)
ax1.set_xlabel('Illuminant')
ax1.set_ylabel('Algorithm')
ax1.set_title('Primary Illuminant White Patch Errors')
ax1.set_xticks(range(NUM_ILLUMINANTS))
ax1.set_xticklabels(ILLUMINANTS)
ax1.set_yticks(range(num_methods))
ax1.set_yticklabels([f'{cfg.awb_method} {cfg.region_method[0]}' for cfg in ALL_CONFIGS])
plt.colorbar(im1, ax=ax1, label='Euclidean Distance')

# 2. Heatmap for small patches
ax2 = axes[0, 1]
im2 = ax2.imshow(illuminant_stats[:, :, 1].T, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.5)
ax2.set_xlabel('Illuminant')
ax2.set_ylabel('Algorithm')
ax2.set_title('Secondary Illuminant White Patch Errors')
ax2.set_xticks(range(NUM_ILLUMINANTS))
ax2.set_xticklabels(ILLUMINANTS)
ax2.set_yticks(range(num_methods))
ax2.set_yticklabels([f'{cfg.awb_method} {cfg.region_method[0]}' for cfg in ALL_CONFIGS])
plt.colorbar(im2, ax=ax2, label='Euclidean Distance')

# 3. Per-illuminant error across methods (big patches)
ax3 = axes[1, 0]
for m_idx in range(num_methods):
    ax3.plot(range(NUM_ILLUMINANTS), illuminant_stats[:, m_idx, 0], 
             marker='o', label=f'Method {ALL_CONFIGS[m_idx].awb_method}')
ax3.set_xlabel('Illuminant Index')
ax3.set_ylabel('Euclidean Distance')
ax3.set_title('Big Patch: Error per Illuminant')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Statistical summary per illuminant
ax4 = axes[1, 1]
illuminant_mean_error = np.mean(illuminant_stats[:, :, 0], axis=1)  # Average across methods
illuminant_std_error = np.std(illuminant_stats[:, :, 0], axis=1)
x_pos = np.arange(NUM_ILLUMINANTS)
ax4.bar(x_pos, illuminant_mean_error, yerr=illuminant_std_error, capsize=5, alpha=0.7)
ax4.set_xlabel('Illuminant Index')
ax4.set_ylabel('Mean Error (across methods)')
ax4.set_title('Big Patch: Illuminant Difficulty')
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Identify most challenging illuminants
print("\n=== Per-Illuminant Statistics (Big Patch) ===")
for ill_idx in range(NUM_ILLUMINANTS):
    mean_err = np.mean(illuminant_stats[ill_idx, :, 0])
    std_err = np.std(illuminant_stats[ill_idx, :, 0])
    worst_method = np.argmax(illuminant_stats[ill_idx, :, 0])
    best_method = np.argmin(illuminant_stats[ill_idx, :, 0])
    
    print(f"Illuminant {ill_idx}: Mean={mean_err:.4f}, Std={std_err:.4f}, "
          f"Worst=Method {worst_method} ({illuminant_stats[ill_idx, worst_method, 0]:.4f}), "
          f"Best=Method {best_method} ({illuminant_stats[ill_idx, best_method, 0]:.4f})")

# Find worst-case scenarios
print("\n=== Worst Case Scenarios ===")
worst_cases = []
for ill_idx in range(NUM_ILLUMINANTS):
    for m_idx in range(num_methods):
        error = illuminant_stats[ill_idx, m_idx, 0]
        if error > 0.2:  # Threshold for "bad"
            worst_cases.append((ill_idx, m_idx, error))

worst_cases.sort(key=lambda x: x[2], reverse=True)
for ill_idx, m_idx, error in worst_cases[:10]:  # Top 10 worst
    print(f"Illuminant {ill_idx}, Method {m_idx}: Error = {error:.4f}")

# Performance on Primary vs Secondary Illuminant

In [ ]:
# Better visualization showing the multi-illuminant challenge
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Multi-Illuminant AWB Performance on Primary vs Secondary Illuminant', fontsize=16)

# 1. Direct comparison: Big vs Small
ax1 = axes[0, 0]
all_methods = [cfg.awb_method for cfg in ALL_CONFIGS]
x = np.arange(len(all_methods))
width = 0.35

big_means = [np.mean(illuminant_stats[:, m, 0]) for m in range(num_methods)]
small_means = [np.mean(illuminant_stats[:, m, 1]) for m in range(num_methods)]

bars1 = ax1.bar(x - width/2, big_means, width, label='Primary Illuminant', alpha=0.8)
bars2 = ax1.bar(x + width/2, small_means, width, label='Secondary Illuminant', alpha=0.8)

ax1.set_ylabel('Mean Euclidean Distance')
ax1.set_title('Primary vs Secondary Illuminant Performance')
ax1.set_xticks(x)
ax1.set_xticklabels(all_methods)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Degradation factor
ax2 = axes[0, 1]
degradation = [(small_means[i] / big_means[i]) for i in range(num_methods)]
colors_deg = ['green' if d < 3 else 'orange' if d < 4 else 'red' for d in degradation]
bars = ax2.bar(x, degradation, color=colors_deg, alpha=0.7)
ax2.axhline(y=1, color='black', linestyle='--', label='No degradation')
ax2.set_ylabel('Error Ratio (Secondary/Primary)')
ax2.set_title('Multi-Illuminant Degradation Factor')
ax2.set_xticks(x)
ax2.set_xticklabels(all_methods)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, degradation):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:.1f}x', ha='center', va='bottom')

# 3. Per-illuminant comparison
ax3 = axes[1, 0]
for m_idx in range(num_methods):
    primary = illuminant_stats[:, m_idx, 0]
    secondary = illuminant_stats[:, m_idx, 1]
    ax3.scatter(primary, secondary, label=f'Method {all_methods[m_idx]}', s=100, alpha=0.7)

# Add diagonal line (equal performance)
max_val = max(ax3.get_xlim()[1], ax3.get_ylim()[1])
ax3.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Equal performance')
ax3.set_xlabel('Primary Illuminant Error')
ax3.set_ylabel('Secondary Illuminant Error')
ax3.set_title('Error Correlation: Primary vs Secondary')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Worst-case analysis
ax4 = axes[1, 1]
# Find which method-illuminant combinations have worst secondary performance
worst_secondary = []
for ill_idx in range(NUM_ILLUMINANTS):
    for m_idx in range(num_methods):
        err = illuminant_stats[ill_idx, m_idx, 1]
        worst_secondary.append((ill_idx, m_idx, err))

worst_secondary.sort(key=lambda x: x[2], reverse=True)
top_n = 8
ill_indices = [w[0] for w in worst_secondary[:top_n]]
method_indices = [w[1] for w in worst_secondary[:top_n]]
errors = [w[2] for w in worst_secondary[:top_n]]
labels = [f'Ill{ill_indices[i]}/M{method_indices[i]}' for i in range(top_n)]

bars = ax4.barh(range(top_n), errors, color='darkred', alpha=0.7)
ax4.set_yticks(range(top_n))
ax4.set_yticklabels(labels)
ax4.set_xlabel('Error (Secondary Illuminant)')
ax4.set_title('Worst Secondary Illuminant Failures')
ax4.grid(axis='x', alpha=0.3)
ax4.invert_yaxis()

plt.tight_layout()
plt.show()

# plot results

In [ ]:
import src.test as test
import src.methods as methods
from src.utils import isp
import src.utils.config as config
import numpy as np
import matplotlib.pyplot as plt

alg_names = {0: 'GW',
             1: 'LGW',
             2: 'CLGW',
             3: 'CCT-AWB'}

reg_names = {'superpixels': 'SP',
             'tiles': 'T'}

def plot_results_minimal(mode: str, cfg: config.AWBConfig, show_plots: bool = True, save_file: str = None):
    wps_big = np.zeros((test.NUM_ILLUMINANTS, 3))
    wps_small = np.zeros((test.NUM_ILLUMINANTS, 3))

    fig, axs = plt.subplots(test.NUM_ILLUMINANTS, 3, figsize=(13, 15))
    # fig.suptitle(f'AWB Results - Algorithm {alg_names[cfg.awb_method]} {reg_names[cfg.region_method]}', fontsize=16)

    for ill_idx, ill in enumerate(test.ILLUMINANTS):
        img = test.get_image(ill_idx, mode)

        result = methods.run_awb(img, cfg, cfg.awb_method)
        img_awb = result['image']
        img_corr = isp.gamma_correction(img_awb)
            
        # Plot image and gain map
        axs[ill_idx, 0].imshow(img_corr)
        axs[ill_idx, 0].set_title(f'Algorithm {alg_names[cfg.awb_method]} {reg_names[cfg.region_method]} on illuminant {test.ILLUMINANTS[ill_idx]}')
        axs[ill_idx, 0].axis('off')

        gain_map = result['smoothed_gain_map']
        gain_map_vis = gain_map / np.max(gain_map)
        axs[ill_idx, 1].imshow(gain_map_vis)
        axs[ill_idx, 1].set_title('Smoothed Gain Map')
        axs[ill_idx, 1].axis('off')
        
        
        # Plot cluster centers and region illuminants
        axs[ill_idx, 2].set_title('Estimated illuminants')
        cct = test.get_updated_cct()
        axs[ill_idx, 2].scatter(cct[:, 0], cct[:, 1], c='gray', marker='o', label='CCT Illuminants', s=40, alpha=0.5)

        # Annotate CCT values
        cct_strings = test.ALL_ILLUMINANTS
        for i, txt in enumerate(cct_strings):
            axs[ill_idx, 2].annotate(txt, (cct[i, 0], cct[i, 1]), textcoords="offset points", xytext=(0,5), ha='center', alpha=0.7, fontsize=8)
        
        # Plot region illuminants (input to clustering) colored by cluster assignment
        region_illuminants = result.get('region_illuminants')
        cluster_labels = result.get('cluster_labels')
        mask_inside = result.get('mask_inside')  # Get which regions were inside CCT
        
        if region_illuminants is not None and len(region_illuminants) > 0:
            if cluster_labels is not None and len(cluster_labels) == len(region_illuminants):
                # Filter to only show regions that were inside CCT curve (actually clustered)
                if mask_inside is not None:
                    # Plot regions inside CCT (clustered)
                    valid_regions_inside = mask_inside & ~np.isnan(region_illuminants).any(axis=1)
                    
                    if np.any(valid_regions_inside):
                        region_illums_inside = region_illuminants[valid_regions_inside]
                        cluster_labels_inside = cluster_labels[valid_regions_inside]
                        
                        # Color by cluster assignment
                        unique_labels = np.unique(cluster_labels_inside)
                        colors = ['red', 'green']
                        for idx, label in enumerate(unique_labels):
                            mask = cluster_labels_inside == label
                            axs[ill_idx, 2].scatter(region_illums_inside[mask, 0], region_illums_inside[mask, 1], 
                                                  c=[colors[idx]], label=f'Cluster {label}', 
                                                  s=60, alpha=0.5, marker='.')
                    
                    # Plot regions outside CCT tolerance (not clustered)
                    valid_regions_outside = ~mask_inside & ~np.isnan(region_illuminants).any(axis=1)
                    
                    if np.any(valid_regions_outside):
                        region_illums_outside = region_illuminants[valid_regions_outside]
                        axs[ill_idx, 2].scatter(region_illums_outside[:, 0], region_illums_outside[:, 1], 
                                              c='gray', label=f'Outside CCT ({np.sum(valid_regions_outside)})', 
                                              s=40, alpha=0.6, marker='x')
                else:
                    # No mask_inside available, plot all
                    unique_labels = np.unique(cluster_labels)
                    colors = ['red', 'green']
                    for idx, label in enumerate(unique_labels):
                        mask = cluster_labels == label
                        axs[ill_idx, 2].scatter(region_illuminants[mask, 0], region_illuminants[mask, 1], 
                                              c=[colors[idx]], label=f'Cluster {label}', 
                                              s=60, alpha=0.5, marker='.')
            else:
                # Fallback if no labels available
                axs[ill_idx, 2].scatter(region_illuminants[:, 0], region_illuminants[:, 1], c='lightblue', 
                                      label='Region illuminants', s=50, alpha=0.6, marker='.')

        # Plot cluster centers
        cluster_centers = result.get('cluster_centers')
        if cluster_centers is not None and len(cluster_centers) > 0:
            axs[ill_idx, 2].scatter(cluster_centers[:, 0], cluster_centers[:, 1], c='blue',label='Cluster centers', s=10, marker='o', linewidths=1.5)
        
        gt_ill_1_rb = cct[ill_idx]
        gt_ill_2_rb = test.small_light_ill
        axs[ill_idx, 2].scatter(gt_ill_1_rb[0], gt_ill_1_rb[1], c='black', marker='x', s=50, label=f'GT {test.ILLUMINANTS[ill_idx]}', linewidths=1.5)
        if mode == 'multi':
            axs[ill_idx, 2].scatter(gt_ill_2_rb[0], gt_ill_2_rb[1], c='black', marker='d', s=20, label='GT S')
        axs[ill_idx, 2].set_xlim(0.2, 1.4)
        axs[ill_idx, 2].set_ylim(0.1, 1)
        axs[ill_idx, 2].legend(fontsize=8, loc='upper right')
        axs[ill_idx, 2].grid(alpha=0.3)
        axs[ill_idx, 2].set_xlabel('R/G')
        axs[ill_idx, 2].set_ylabel('B/G')
        axs[ill_idx, 2].set_box_aspect(img_awb.shape[0] / img_awb.shape[1])  # H / W)

        # Compute white points on the AWB corrected image (not gamma corrected!)
        if mode == 'multi':
            wps_small[ill_idx] = test.get_wp_multi_small(img_awb, ill_idx)
            wps_big[ill_idx] = test.get_wp_multi_big(img_awb, ill_idx)
        elif mode == 'single':
            wps_big[ill_idx] = test.get_wp_single(img_awb, ill_idx)
        
        # # # Plot white point estimation in (R/G, B/G) space
        # axs[ill_idx, 3].set_title('White Point Estimation')
        # if mode == 'multi':
        #     axs[ill_idx, 3].scatter(wps_small[ill_idx,0]/wps_small[ill_idx,1], wps_small[ill_idx,2]/wps_small[ill_idx,1], c='blue', label='WP secondary', s=100)
        #     axs[ill_idx, 3].scatter(wps_big[ill_idx,0]/wps_big[ill_idx,1], wps_big[ill_idx,2]/wps_big[ill_idx,1], c='black', label='WP primary', s=100)
        # elif mode == 'single':
        #     axs[ill_idx, 3].scatter(wps_big[ill_idx,0]/wps_big[ill_idx,1], wps_big[ill_idx,2]/wps_big[ill_idx,1], c='black', label='WP primary', s=100)
        # axs[ill_idx, 3].set_xlim(0.4, 1.8)
        # axs[ill_idx, 3].set_ylim(0.4, 1.8)

        # # Pure white point for reference
        # pure_wp = np.array([1.0, 1.0, 1.0])
        # axs[ill_idx, 3].scatter(pure_wp[0]/pure_wp[1], pure_wp[2]/pure_wp[1], c='red', label='Pure neutral', s=100, marker='x')
        # axs[ill_idx, 3].legend()
        # axs[ill_idx, 3].grid()

    plt.tight_layout()

    if save_file is not None:
        plt.savefig(save_file, dpi=300, bbox_inches='tight')
    if show_plots:
        plt.show()
    plt.close(fig)
    return wps_big, wps_small

# ------------------------------------------------------------------

base_cfg = config.AWBConfig(
    debug=False,
    awb_method=2,
    num_tiles=10,
    num_superpixels=30,
    mask_saturation_threshold=0.9999,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.1,
    downsample=True,
    downsample_scale=0.2,
    region_method='tiles',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.05,
    smooth1_method='none',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-4},
    smooth2_method='none',
    smooth2_kwargs={},
)
cfg = base_cfg
mode = 'multi'  # 'single', 'multi'
wps_big, wps_small = plot_results_minimal(mode, cfg, show_plots=True, save_file="test_awb_result.pdf")

# saving all the result plots

In [ ]:
from cfgs import ALL_CONFIGS

for cfg in ALL_CONFIGS:
    print(f'Plotting results for Algorithm {cfg.awb_method} with region method {cfg.region_method}')

    for mode in ['single', 'multi']:
        save_filename = f'results/plot_new/plot_{mode}_{cfg.awb_method}_{cfg.region_method}_results.pdf'

        plot_results_minimal(mode, cfg, show_plots=False, save_file=save_filename)

# FINDING ALL THE BEST ONES

## 0

In [ ]:
from benchmark import benchmark_runtime

base_cfg = config.AWBConfig(
    debug=False,
    awb_method=3,
    num_tiles=20,
    num_superpixels=24,
    mask_saturation_threshold=0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.15,
    downsample=True,
    downsample_scale=0.2,
    region_method='tiles',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-4},
    smooth2_method='none',
    smooth2_kwargs={},
)
cfg = base_cfg
mode = 'multi'  # 'single', 'multi'

#benchmark_runtime(get_image(0, mode), cfg)

wps_big, wps_small = plot_results_minimal(mode, cfg)

distances, avg_distance_big, avg_distance_small, avg_distance = compute_average_distance(wps_big, wps_small)

## 1 t

In [ ]:
cfg = config.AWBConfig(
    debug=False,
    awb_method=1,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.05,
    downsample=True,
    downsample_scale=0.2,
    region_method='tiles',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-4},
    smooth2_method='none',
    smooth2_kwargs={},
)
# benchmark_runtime(get_image(0, mode), cfg)
wps_big, wps_small = plot_results_minimal(mode, cfg)

distances, avg_distance_big, avg_distance_small, avg_distance = compute_average_distance(wps_big, wps_small)

## 1 sp

In [ ]:
cfg = config.AWBConfig(
    debug=False,
    awb_method=1,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.05,
    downsample=True,
    downsample_scale=0.1,
    region_method='superpixels',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-4},
    smooth2_method='none',
    smooth2_kwargs={},
)
# benchmark_runtime(get_image(0, mode), cfg)
wps_big, wps_small = plot_results_minimal(mode, cfg)

## 2 t

In [ ]:
cfg = config.AWBConfig(
    debug=False,
    awb_method=2,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.05,
    downsample=True,
    downsample_scale=0.2,
    region_method='tiles',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-4},
    smooth2_method='none',
    smooth2_kwargs={},
)
# benchmark_runtime(get_image(0, mode), cfg)
wps_big, wps_small = plot_results_minimal(mode, cfg)
wps_big

## 2 sp

In [ ]:
cfg = config.AWBConfig(
    debug=False,
    awb_method=2,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.05,
    downsample=True,
    downsample_scale=0.2,
    region_method='superpixels',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.1,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 10, 'guided_eps': 1e-3},
    smooth2_method='none',
    smooth2_kwargs={},
)
# benchmark_runtime(get_image(0, mode), cfg)
wps_big, wps_small = plot_results_minimal(mode, cfg)

## 3 t

In [ ]:
cfg = config.AWBConfig(
    debug=False,
    awb_method=3,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.97,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.2,
    downsample=True,
    downsample_scale=0.4,
    region_method='tiles',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.1,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-3},
    smooth2_method='none',
    smooth2_kwargs={},
)

cfg = config.AWBConfig(
    debug=False,
    awb_method=3,
    num_tiles=12,
    num_superpixels=24,
    mask_saturation_threshold=0.98,#0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.14,
    downsample=True,
    downsample_scale=0.2,
    region_method='tiles',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.1,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 20, 'guided_eps': 1e-6},
    smooth2_method='none',
    smooth2_kwargs={},
)

wps_big, wps_small = plot_results_minimal(mode, cfg)


0.057102547643744575 0.2181276638367155 0.13761510574023003


## 3 sp

In [ ]:
cfg = config.AWBConfig(
    debug=False,
    awb_method=3,
    num_tiles=10,
    num_superpixels=24,
    mask_saturation_threshold=0.95,
    mask_black_threshold=0.01,
    eps=1e-6,
    cct_curve_rb=test.get_updated_cct(),
    cct_tolerance=0.15,
    downsample=True,
    downsample_scale=0.1,
    region_method='superpixels',
    sp_compactness=10,
    cluster_method='kmeans',
    cluster_kwargs={},
    cluster_min_center_distance=0.02,
    cluster_min_ratio=0.05,
    smooth1_method='guided',
    smooth1_kwargs={'guided_radius': 10, 'guided_eps': 1e-4},
    smooth2_method='none',
    smooth2_kwargs={},
)
# benchmark_runtime(get_image(0, mode), cfg)
wps_big, wps_small = plot_results_minimal(mode, cfg)

# all white points

In [ ]:
cfg = base_cfg

mode = 'multi'  # 'single', 'multi'

# Plot all white point results on a single plot
plt.figure(figsize=(10, 8))
plt.title('White Point Estimations for All Illuminants')

# Define colors for each method
colors = plt.cm.tab10(np.linspace(0, 1, NUM_METHODS))

for m in range(NUM_METHODS):
    for reg in ['t']:
    # for reg in REGIONS:
        cfg.awb_method = m
        cfg.region_method = 'superpixels' if reg == 'sp' else 'tiles'

        wps_small = np.zeros((NUM_ILLUMINANTS, 3))
        wps_big = np.zeros((NUM_ILLUMINANTS, 3))

        for ill_idx, ill in enumerate(ILLUMINANTS):
            img = get_image(ill_idx, mode)

            result = methods.run_awb(img, cfg, cfg.awb_method)
            img_awb = result['image']
            
            if mode == 'multi':
                wps_small[ill_idx] = test.get_wp_multi_small(img_awb, ill_idx)
                wps_big[ill_idx] = test.get_wp_multi_big(img_awb, ill_idx)
            elif mode == 'single':
                wps_big[ill_idx] = test.get_wp_single(img_awb, ill_idx)

        # Use different markers for different regions/patch sizes
        if mode == 'multi':
            marker_small = 'o' if reg == 'sp' else 's'
            marker_big = '^' if reg == 'sp' else 'D'
            
            plt.scatter(wps_small[:,0]/wps_small[:,1], wps_small[:,2]/wps_small[:,1], 
                       c=[colors[m]], label=f'Method {m} small ({reg})', s=100, marker=marker_small)
            plt.scatter(wps_big[:,0]/wps_big[:,1], wps_big[:,2]/wps_big[:,1], 
                       c=[colors[m]], label=f'Method {m} big ({reg})', s=100, marker=marker_big)
        elif mode == 'single':
            marker = 'o' if reg == 'sp' else 's'
            plt.scatter(wps_big[:,0]/wps_big[:,1], wps_big[:,2]/wps_big[:,1], 
                       c=[colors[m]], label=f'Method {m} ({reg})', s=100, marker=marker)

# Pure white point for reference
pure_wp = np.array([1.0, 1.0, 1.0])
plt.scatter(pure_wp[0]/pure_wp[1], pure_wp[2]/pure_wp[1], c='black', label='Pure Neutral', s=200, marker='x', linewidths=3)

plt.xlim(0.5, 1.5)
plt.ylim(0.5, 1.5)
plt.xlabel('R/G')
plt.ylabel('B/G')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# cluster analysis

In [ ]:
import pandas as pd
from src import methods

def cluster_analysis(cfg, mode='multi'):
    # Debug clustering for all illuminants
    print("\n" + "="*80)
    print("CLUSTER ANALYSIS")
    print("="*80)

    for ill_idx in range(NUM_ILLUMINANTS):
        img = get_image(ill_idx, mode)
        result = methods.run_awb(img, cfg, cfg.awb_method)
        
        cluster_centers = result.get('cluster_centers')
        cluster_labels = result.get('cluster_labels')
        
        print(f"\n{ILLUMINANTS[ill_idx]}:")
        print(f"  Num cluster centers: {len(cluster_centers) if cluster_centers is not None else 'None'}")
        
        if cluster_labels is not None and cluster_centers is not None and len(cluster_centers) > 1:
            unique, counts = np.unique(cluster_labels, return_counts=True)
            total = counts.sum()
            ratios = counts / total
            print(f"  Cluster sizes: {counts} | Ratios: {ratios} | Min ratio: {ratios.min():.4f}")
            print(f"  Center distance: {np.linalg.norm(cluster_centers[0] - cluster_centers[1]):.4f}")
        elif cluster_centers is not None and len(cluster_centers) == 1:
            print(f"  Fallback to single illuminant")
        else:
            print(f"  No clusters found")
            continue

        distances_to_ground_truth_illuminants = []
        if cluster_centers is not None:
            gt_ill_1_rb = test.get_updated_cct()[ill_idx]
            gt_ill_2_rb = test.small_light_ill
            for i in range(len(cluster_centers)):
                dist1 = np.linalg.norm(cluster_centers[i] - gt_ill_1_rb)
                dist2 = np.linalg.norm(cluster_centers[i] - gt_ill_2_rb)
                dist = min(dist1, dist2)
                distances_to_ground_truth_illuminants.append(dist)
    return distances_to_ground_truth_illuminants

from cfgs import ALL_CONFIGS_TILE

dist = cluster_analysis(ALL_CONFIGS_TILE[-1], mode='multi')

In [ ]:
from src import methods
from src import test

def cluster_analysis(cfg, method_label=0, mode='multi'):
    rows = []

    for ill_idx in range(test.NUM_ILLUMINANTS):
        img = test.get_image(ill_idx, mode)
        result = methods.run_awb(img, cfg, cfg.awb_method)
        
        cluster_centers = result.get('cluster_centers')
        cluster_labels = result.get('cluster_labels')

        ill_name = test.ALL_ILLUMINANTS[ill_idx]

        if cluster_labels is not None and cluster_centers is not None and len(cluster_centers) > 1:
            unique, counts = np.unique(cluster_labels, return_counts=True)
            total = counts.sum()
            ratios = counts / total
            min_ratio = ratios.min()
            center_dist = np.linalg.norm(cluster_centers[0] - cluster_centers[1])
        else:
            min_ratio = None
            center_dist = None

        # Ground truth
        gt1 = test.get_updated_cct()[ill_idx]
        gt2 = test.small_light_ill

        if cluster_centers is not None:
            c1 = cluster_centers[0]
            d_c1_gt1 = np.linalg.norm(c1 - gt1)
            d_c1_gt2 = np.linalg.norm(c1 - gt2)

            if len(cluster_centers) > 1:
                c2 = cluster_centers[1]
                d_c2_gt1 = np.linalg.norm(c2 - gt1)
                d_c2_gt2 = np.linalg.norm(c2 - gt2)
            else:
                d_c2_gt1 = None
                d_c2_gt2 = None
        else:
            d_c1_gt1 = d_c1_gt2 = None
            d_c2_gt1 = d_c2_gt2 = None

        # -------- NEW: closest-cluster distances --------
        if d_c2_gt1 is not None:
            dist_to_gt1 = min(d_c1_gt1, d_c2_gt1)
            dist_to_gt2 = min(d_c1_gt2, d_c2_gt2)
        else:
            dist_to_gt1 = d_c1_gt1
            dist_to_gt2 = d_c1_gt2
        # -----------------------------------------------

        rows.append({
            "method_label": method_label,
            "method": cfg.awb_method,
            "region": cfg.region_method,
            "illuminant": ill_name,
            "num_clusters": len(cluster_centers) if cluster_centers is not None else 0,
            "min_cluster_ratio": min_ratio,
            "center_distance": center_dist,

            # "c1_to_gt1": d_c1_gt1,
            # "c1_to_gt2": d_c1_gt2,
            # "c2_to_gt1": d_c2_gt1,
            # "c2_to_gt2": d_c2_gt2,

            # NEW FINAL METRICS
            "dist_to_gt_primary": dist_to_gt1,
            "dist_to_gt_secondary": dist_to_gt2
        })

    return rows

# ------------------------------------------------
from cfgs import ALL_CONFIGS

for mode in ['single', 'multi']:

    all_rows = []

    for idx, cfg in enumerate(ALL_CONFIGS):
        if cfg.awb_method in [1]:
            continue
        all_rows.extend(cluster_analysis(cfg, method_label=f'{cfg.awb_method}{cfg.region_method[0]}', mode=mode))

    df = pd.DataFrame(all_rows)
    df = df.round(4)
    df.to_csv(f"results/cluster_analysis_{mode}.csv", index=False)

    # print(df)



In [ ]:
# Summarize results
df = pd.read_csv(f"results/cluster_analysis_multi.csv")
summary = df.groupby("method_label").agg(
    mean_dist_prim = ("dist_to_gt_primary","mean"),
    std_dist_prim  = ("dist_to_gt_primary","std"),
    mean_dist_sec = ("dist_to_gt_secondary","mean"),
    std_dist_sec  = ("dist_to_gt_secondary","std")
).round(4)

summary.to_csv("results/cluster_analysis_summary_results.csv")

from results.csv_to_latex import csv_to_latex_table

csv_to_latex_table("results/cluster_analysis_summary_results.csv", 
    caption="Summary of Cluster Analysis Results", 
    label="tab:cluster_analysis_summary_results", 
    float_format=".3f")

## closest illuminant

In [ ]:
import numpy as np
def nearest_cct_label(pt_rb: np.ndarray, cct_curve: np.ndarray):
    """Return index of nearest CCT point for given RB point."""
    dists = np.linalg.norm(cct_curve - pt_rb[None, :], axis=1)
    idx = int(np.argmin(dists))
    return idx, float(dists[idx])

import pandas as pd
from cfgs import ALL_CONFIGS
from src.methods import run_awb
import src.test as test

rows = []

ills = test.ALL_ILLUMINANTS
cct = test.get_updated_cct()

for cfg in ALL_CONFIGS:
    for ill_idx in range(test.NUM_ILLUMINANTS):

        if cfg.awb_method == 1:
            # Do NOT evaluate illuminant identity
            # Only evaluate gain-map / white-point metrics
            continue

        img = test.get_image(ill_idx, 'multi')

        result = run_awb(img, cfg, cfg.awb_method)
        ests = result['cluster_centers']

        closest_ill1 = None
        closest_ill2 = None
        dist1 = None
        dist2 = None
        idx1 = None
        idx2 = None
        small_ill_idx = 5  # Index of small illuminant in ALL_ILLUMINANTS
    
        idx1, dist1 = nearest_cct_label(ests[0], cct)
        closest_ill1 = ills[idx1]

        if len(ests) > 1:
            idx2, dist2 = nearest_cct_label(ests[1], cct)
            closest_ill2 = ills[idx2]
        
            primary_correct = (idx1 == ill_idx) or (idx2 == ill_idx)
            secondary_correct = (idx1 == small_ill_idx) or (idx2 == small_ill_idx)
        else:
            primary_correct = (idx1 == ill_idx)
            secondary_correct = (idx1 == small_ill_idx)

        rows.append({
            "method": cfg.awb_method,
            "region": cfg.region_method,
            "gt_primary_ill": ills[ill_idx],
            "closest_ill1": closest_ill1,
            "closest_ill2": closest_ill2,
            # "dist1": dist1,
            # "dist2": dist2
            "primary_correct": primary_correct,
            "secondary_correct": secondary_correct
        })
df = pd.DataFrame(rows)
# print(df)

df.to_csv(f"results/cluster_closest_illuminants.csv", index=False)


from results.csv_to_latex import csv_to_latex_table

csv_to_latex_table("results/cluster_closest_illuminants.csv", 
    caption="Closest CCT Illuminants Identified by Clustering", 
    label="tab:cluster_closest_illuminants", 
    float_format=".2f")

# plot the cluster analysis

## box plot cluster error

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("awb_results_clusters.csv")

fig, ax = plt.subplots()

methods = sorted(df["method_label"].unique())

data_gt1 = [df[df.method_label==m]["dist_to_gt1"] for m in methods]
data_gt2 = [df[df.method_label==m]["dist_to_gt2"] for m in methods]

pos1 = [i*2 for i in range(len(methods))]
pos2 = [i*2+0.7 for i in range(len(methods))]

ax.boxplot(data_gt1, positions=pos1, widths=0.5)
ax.boxplot(data_gt2, positions=pos2, widths=0.5)

ax.set_xticks([p+0.35 for p in pos1])
ax.set_xticklabels(methods)
ax.set_xlabel("Algorithm")
ax.set_ylabel("Distance from estimate to GT")

# ax.legend(["GT1","GT2"])
plt.title("Estimated Illuminant Error Distribution (Left: Primary, Right: Secondary)")
plt.savefig("results/cluster_distance_boxplot.pdf", bbox_inches='tight', dpi=600)
plt.show()


## summary table

In [ ]:
summary = df.groupby("method_label").agg(
    mean_gt1 = ("dist_to_gt_primary","mean"),
    std_gt1  = ("dist_to_gt_primary","std"),
    mean_gt2 = ("dist_to_gt_secondary","mean"),
    std_gt2  = ("dist_to_gt_secondary","std")
).round(4)

summary.to_csv("results/cluster_analysis_summary_results.csv")
print(summary)

from results.csv_to_latex import csv_to_latex_table

csv_to_latex_table("results/cluster_analysis_summary_results.csv", 
    caption="Summary of Cluster Analysis Results", 
    label="tab:cluster_analysis_summary_results", 
    float_format=".4f")

# Runtime

In [ ]:
import numpy as np
import pandas as pd
import src.utils.isp as isp
from benchmark import benchmark_runtime
from cfgs import ALL_CONFIGS

# Load a sample image
img_file = 'capture/lightbox/img_multi_D65.npy'
img = np.load(img_file)
img = isp.black_level_correction(img)
img = isp.demosaic(img)

all_rows = []
for idx, cfg in enumerate(ALL_CONFIGS):
    print(f'Benchmarking Algorithm {cfg.awb_method} with region method {cfg.region_method}')
    all_rows.extend(benchmark_runtime(img, cfg, display=False))

df = pd.DataFrame(all_rows)
df = df.round(2)
df.to_csv(f"results/runtime.csv", index=False)

print(df)

from results.csv_to_latex import csv_to_latex_table

csv_to_latex_table("results/runtime.csv", 
    caption="Runtime Results", 
    label="tab:runtime", 
    float_format=".2f")

In [ ]:
import numpy as np
import pandas as pd
import src.utils.isp as isp
from benchmark import benchmark_runtime
from cfgs import base_cfg

# Load a sample image
img_file = 'capture/lightbox/img_multi_D65.npy'
img = np.load(img_file)
img = isp.black_level_correction(img)
img = isp.demosaic(img)

cfg = base_cfg
all_rows = []
for idx in range(4):
    cfg.awb_method = idx
    for region in ['tiles', 'superpixels']:
        cfg.region_method = region
        print(f'Benchmarking Algorithm {cfg.awb_method} with region method {cfg.region_method}')
        all_rows.extend(benchmark_runtime(img, cfg, display=False))
df = pd.DataFrame(all_rows)
df = df.round(2)
df.to_csv(f"results/runtime_with_same_cfg.csv", index=False)

print(df)

from results.csv_to_latex import csv_to_latex_table

csv_to_latex_table("results/runtime_with_same_cfg.csv", 
    caption="Runtime Results", 
    label="tab:runtime_with_same_cfg", 
    float_format=".2f")

# reflectance-dominated regions 

In [ ]:
import numpy as np
import pandas as pd

def rgb_to_rb_rg_div_g(rgb, eps=1e-8):
    """(r,b) with r=R/G, b=B/G."""
    g = np.maximum(rgb[..., 1], eps)
    r = rgb[..., 0] / g
    b = rgb[..., 2] / g
    return np.stack([r, b], axis=-1)

def tile_means(img, num_tiles):
    """Return (T,T,3) mean RGB per tile (cropped to divisible area)."""
    H, W = img.shape[:2]
    T = num_tiles
    Hc = (H // T) * T
    Wc = (W // T) * T
    imgc = img[:Hc, :Wc, :]

    th = Hc // T
    tw = Wc // T

    tiles = imgc.reshape(T, th, T, tw, 3)
    return tiles.mean(axis=(1, 3))

def classify_tiles_by_cct_distance_rg_div_g(img, cct_rb, num_tiles=10, cct_threshold=0.15):
    """
    cct_rb: (K,2) array of CCT points in (r,b) where r=R/G, b=B/G
    Returns: summary dict, dataframe, class map (T,T), dmin map (T,T)
    """
    means_rgb = tile_means(img, num_tiles)                 # (T,T,3)
    tile_rb = rgb_to_rb_rg_div_g(means_rgb)                # (T,T,2)

    # min Euclidean distance to CCT locus samples
    diffs = tile_rb[..., None, :] - cct_rb[None, None, :, :]  # (T,T,K,2)
    dists = np.linalg.norm(diffs, axis=-1)                    # (T,T,K)
    dmin = dists.min(axis=-1)                                 # (T,T)

    t = float(cct_threshold)
    cls = np.zeros_like(dmin, dtype=np.uint8)
    cls[(dmin >= t) & (dmin < 2 * t)] = 1
    cls[dmin >= 2 * t] = 2

    counts = np.bincount(cls.ravel(), minlength=3)
    total = int(cls.size)
    summary = {
        "num_tiles_total": total,
        "illuminance_dominated": int(counts[0]),
        "ambiguous": int(counts[1]),
        "reflectance_dominated": int(counts[2]),
        "reflectance_fraction": float(counts[2] / total),
        "threshold": t,
    }

    # per-tile table
    T = num_tiles
    yy, xx = np.indices((T, T))
    df = pd.DataFrame({
        "tile_y": yy.ravel(),
        "tile_x": xx.ravel(),
        "mean_R": means_rgb[..., 0].ravel(),
        "mean_G": means_rgb[..., 1].ravel(),
        "mean_B": means_rgb[..., 2].ravel(),
        "r": tile_rb[..., 0].ravel(),
        "b": tile_rb[..., 1].ravel(),
        "dmin": dmin.ravel(),
        "class": cls.ravel(),  # 0/1/2
    })
    df["class_name"] = df["class"].map({0: "illuminance", 1: "ambiguous", 2: "reflectance"})

    return summary, df, cls, dmin, tile_rb

def render_tile_mean_image(img, means_rgb, num_tiles):
    """
    img: original HxWx3 image
    means_rgb: (T,T,3) mean RGB per tile
    Returns: HxWx3 image where each tile is filled with its mean color
    """
    H, W = img.shape[:2]
    T = num_tiles

    # crop to divisible area (same as tile_means)
    Hc = (H // T) * T
    Wc = (W // T) * T

    th = Hc // T
    tw = Wc // T

    out = np.zeros((Hc, Wc, 3), dtype=img.dtype)

    for ty in range(T):
        for tx in range(T):
            y0 = ty * th
            y1 = y0 + th
            x0 = tx * tw
            x1 = x0 + tw
            out[y0:y1, x0:x1, :] = means_rgb[ty, tx]

    return out


# ---- usage ----
import numpy as np
import pandas as pd
from src.utils import isp
from src import test

cct_threshold = 0.15
cct = test.get_updated_cct()   # (K,2) in (r,b) with r=R/G, b=B/G
num_tiles = 10

rows = []
for ill_idx in range(test.NUM_ILLUMINANTS):

    img = test.get_image(ill_idx)

    summary, df_tiles, cls_map, dmin_map, tile_rb_map = classify_tiles_by_cct_distance_rg_div_g(
        img, cct_rb=cct, num_tiles=num_tiles, cct_threshold=cct_threshold
    )

    rows.append({
        "illuminant": test.ALL_ILLUMINANTS[ill_idx],
        **summary
    })

df_summary = pd.DataFrame(rows)
print(df_summary)



import matplotlib.pyplot as plt

for ill_idx in range(test.NUM_ILLUMINANTS):
    img = test.get_image(ill_idx)
    means_rgb = tile_means(img, num_tiles)

    tile_mean_img = render_tile_mean_image(
        img,
        means_rgb,
        num_tiles=num_tiles
    )
    plt.figure(figsize=(5,5))
    plt.imshow(np.clip(isp.gamma_correction(tile_mean_img), 0, 1))
    plt.title(f"Tile mean color visualization - Illuminant {test.ALL_ILLUMINANTS[ill_idx]}")
    plt.axis("off")
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CLASS_NAMES = {0: "illum", 1: "ambig", 2: "refl"}

def render_tile_mean_image(img, means_rgb, num_tiles):
    H, W = img.shape[:2]
    T = num_tiles

    Hc = (H // T) * T
    Wc = (W // T) * T
    th = Hc // T
    tw = Wc // T

    out = np.zeros((Hc, Wc, 3), dtype=img.dtype)
    for ty in range(T):
        for tx in range(T):
            y0, y1 = ty * th, (ty + 1) * th
            x0, x1 = tx * tw, (tx + 1) * tw
            out[y0:y1, x0:x1, :] = means_rgb[ty, tx]
    return out

def render_class_heatmap(cls_map, tile_shape, colors=None):
    """
    cls_map: (T,T) with {0,1,2}
    tile_shape: (th, tw) tile pixel size for expansion
    colors: dict {0: (r,g,b), 1: (r,g,b), 2: (r,g,b)} in [0,1]
    Returns: (T*th, T*tw, 3) RGB heatmap aligned to tile_mean_img
    """
    if colors is None:
        # 3-color palette (safe defaults)
        colors = {
            0: (0.20, 0.80, 0.20),  # illuminance: green
            1: (0.95, 0.80, 0.20),  # ambiguous: yellow
            2: (0.90, 0.20, 0.20),  # reflectance: red
        }

    T = cls_map.shape[0]
    th, tw = tile_shape
    out = np.zeros((T * th, T * tw, 3), dtype=np.float32)

    for ty in range(T):
        for tx in range(T):
            y0, y1 = ty * th, (ty + 1) * th
            x0, x1 = tx * tw, (tx + 1) * tw
            out[y0:y1, x0:x1, :] = colors[int(cls_map[ty, tx])]
    return out

def plot_tile_overlay_with_labels(tile_mean_img, cls_map, alpha=0.35, colors=None, text_color="white"):
    """
    Shows:
      1) tile-mean image with a semi-transparent 3-color overlay
      2) class labels (illum/ambig/refl) centered per tile
    """
    Hc, Wc = tile_mean_img.shape[:2]
    T = cls_map.shape[0]
    th = Hc // T
    tw = Wc // T

    heat = render_class_heatmap(cls_map, (th, tw), colors=colors)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.clip(tile_mean_img, 0, 1))
    ax.imshow(heat, alpha=alpha)

    # grid + labels
    for ty in range(T):
        for tx in range(T):
            y0 = ty * th
            x0 = tx * tw
            ax.add_patch(Rectangle((x0, y0), tw, th, fill=False, linewidth=0.8))
            name = CLASS_NAMES[int(cls_map[ty, tx])]
            ax.text(
                x0 + tw / 2, y0 + th / 2, name,
                ha="center", va="center",
                color=text_color, fontsize=9
            )

    ax.set_title("Tile means + class overlay + labels")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

def plot_class_heatmap_only(tile_mean_img, cls_map, colors=None, show_grid=True):
    """
    3-color class heatmap aligned to the same tile grid (no labels).
    """
    Hc, Wc = tile_mean_img.shape[:2]
    T = cls_map.shape[0]
    th = Hc // T
    tw = Wc // T

    heat = render_class_heatmap(cls_map, (th, tw), colors=colors)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(heat)
    if show_grid:
        for ty in range(T):
            for tx in range(T):
                ax.add_patch(Rectangle((tx * tw, ty * th), tw, th, fill=False, linewidth=0.8))
    ax.set_title("3-color class heatmap (aligned to tiles)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# ---- usage (end-to-end with your existing outputs) ----
# Assuming you already ran:
# summary, df_tiles, cls_map, dmin_map, tile_rb_map = classify_tiles_by_cct_distance_rg_div_g(...)
# and you can recompute/keep means_rgb:

means_rgb = tile_means(img, num_tiles)
tile_mean_img = render_tile_mean_image(img, means_rgb, num_tiles=num_tiles)

# 1) Overlay with labels on top of tile-mean image
plot_tile_overlay_with_labels(tile_mean_img, cls_map, alpha=0.35)

# 2) Heatmap only (aligned)
plot_class_heatmap_only(tile_mean_img, cls_map, show_grid=True)
